# Understanding CT Registration — From Scratch

## The big picture problem

You have a dog that gets a **planning CT** (high quality, Siemens scanner) on day 1.
Then over the next few days of treatment, the dog gets **cone-beam CTs (CBCTs)** on the treatment machine (Varian).

The problem: **the dog isn't in the exact same position each time.** It shifts a little, rotates a little.

The RE files contain the answer: "here's exactly how the dog moved between scans."
They store a **4x4 matrix** that says how to transform one scan's coordinates into another's.

Let's build up to understanding that, one step at a time.

In [17]:
import pydicom
import numpy as np
from pathlib import Path

## Step 1: What files do we actually have?

Before doing anything fancy, let's just look at what's in a patient folder.
Each file starts with a prefix like CT, RE, RS, etc. — that tells you what kind of DICOM file it is.

In [ ]:
# Point this at one patient — we'll use Martin as our example
patient_dir = Path("../data/new_Dog_CT/Martin, Odin/Martin, Odin")

# Count files by their prefix (the part before the first dot)
from collections import Counter
prefixes = Counter()
for f in patient_dir.glob("*.dcm"):
    prefix = f.name.split(".")[0]
    prefixes[prefix] += 1

for prefix, count in prefixes.most_common():
    print(f"  {prefix:4s} -> {count} files")

  CT   -> 735 files
  RT   -> 6 files
  RS   -> 5 files
  RI   -> 5 files
  RE   -> 4 files
  RP   -> 1 files
  RD   -> 1 files


## What are all these file types?

- **CT** — the actual image slices. Each file = one 2D slice. A bunch of them stacked = a 3D volume.
- **RE** — "Registration". This is the one we care about most. It stores the transform to align two scans.
- **RS** — structure sets (contours drawn around tumors/organs). Not needed for alignment.
- **RT/RI/RD/RP** — treatment-related stuff (plans, doses, beam images). Not needed for alignment.

So really, we only care about **CT** files and **RE** files for this task.

## Step 2: There are 735 CT files but they're NOT all from the same scan

This is the key thing to understand. Multiple scans were taken on different days.
DICOM groups slices from the same scan under a **Series Instance UID**.
Let's see what series we have.

In [19]:
# Read just the header of every CT file (skip pixel data — way faster)
# Group them by Series Instance UID
from collections import defaultdict

series = defaultdict(list)
for f in sorted(patient_dir.glob("CT.*.dcm")):
    ds = pydicom.dcmread(f, stop_before_pixels=True)
    series[str(ds.SeriesInstanceUID)].append(f)

print(f"Found {len(series)} separate CT series:\n")
for uid, files in series.items():
    # Read one file to get info about this series
    ds = pydicom.dcmread(files[0], stop_before_pixels=True)
    print(f"  {len(files):3d} slices | date: {ds.SeriesDate} | name: {getattr(ds, 'SeriesDescription', '(none)')}")
    print(f"       machine: {getattr(ds, 'Manufacturer', '?')} | kVP: {getattr(ds, 'KVP', '?')}")
    print()

Found 5 separate CT series:

   86 slices | date: 20250926 | name: (none)
       machine: Varian Medical Systems | kVP: 100

   86 slices | date: 20250924 | name: (none)
       machine: Varian Medical Systems | kVP: 100

   86 slices | date: 20250925 | name: (none)
       machine: Varian Medical Systems | kVP: 100

  239 slices | date: 20250917 | name: ST PRE
       machine: SIEMENS | kVP: 140

  238 slices | date: 20250917 | name: ST POST
       machine: SIEMENS | kVP: 140



## What you should see

- **2 big series** (~239, ~238 slices) from SIEMENS on 2025-09-17 — these are the planning CT ("ST PRE" and "ST POST"). High quality, taken the same day.
- **3 small series** (86 slices each) from Varian on different dates — these are the CBCTs taken during treatment. Lower quality, taken on the treatment machine before each treatment session.

The goal: **align each CBCT to the planning CT** so we can compare them in the same coordinate space.

## Step 3: What's in an RE file?

Now let's open one RE file and look at what's inside.
Don't worry about understanding everything — we'll focus on the important parts.

In [20]:
# Pick one RE file
re_files = sorted(patient_dir.glob("RE.*.dcm"))
print(f"There are {len(re_files)} RE files.\n")

# Let's read one that connects a CBCT to the planning CT
# (the ones with 2 referenced series — we'll find one)
for re_path in re_files:
    ds = pydicom.dcmread(re_path)
    n_ref = len(getattr(ds, "ReferencedSeriesSequence", []))
    if n_ref == 2:
        print(f"Using: {re_path.name}")
        print(f"Modality: {ds.Modality}")
        print(f"SOP Class: {ds.SOPClassUID.name}")
        break

There are 4 RE files.

Using: RE.1.2.246.352.61.10.5054154996046669765.16146340506122103451.dcm
Modality: REG
SOP Class: Spatial Registration Storage


## Step 4: The RE file has two key pieces of info

**Piece 1: "Referenced Series"** — which two CT series does this registration connect?

**Piece 2: "Registration Sequence"** — the actual transform (a 4x4 matrix)

Let's look at piece 1 first.

In [21]:
# Piece 1: Which CT series does this RE file connect?
print("Referenced Series (the two CT scans this registration links):\n")
for ref in ds.ReferencedSeriesSequence:
    ref_uid = str(ref.SeriesInstanceUID)
    n_slices = len(ref.ReferencedInstanceSequence)

    # Try to match this to one of our CT series
    matched = series.get(ref_uid)
    if matched:
        sample = pydicom.dcmread(matched[0], stop_before_pixels=True)
        name = getattr(sample, "SeriesDescription", "(none)")
        date = sample.SeriesDate
        print(f"  Series '{name}' from {date} ({n_slices} slices referenced)")
    else:
        print(f"  Unknown series ({n_slices} slices) — might be from a different timepoint")

Referenced Series (the two CT scans this registration links):

  Series '(none)' from 20250925 (86 slices referenced)
  Series 'ST PRE' from 20250917 (139 slices referenced)


## Step 5: The 4x4 transform matrix — what does it actually mean?

Each RE file has a **RegistrationSequence** with **2 entries**:
- One for the **fixed scan** (the reference — doesn't move). It gets an **identity matrix** (= "do nothing").
- One for the **moving scan** (the one being aligned). It gets the **actual transform**.

Think of it like this: if you put a sticker on the dog's nose, the matrix tells you
"the nose moved from position A in the CBCT to position B in the planning CT."

In [22]:
# Piece 2: Let's look at both registration entries
for i, reg in enumerate(ds.RegistrationSequence):
    print(f"--- Registration entry {i} ---")
    print(f"  Frame of Reference UID: ...{str(reg.FrameOfReferenceUID)[-30:]}")

    # The matrix lives inside MatrixRegistrationSequence -> MatrixSequence
    for mat_reg in getattr(reg, "MatrixRegistrationSequence", []):
        for mat in getattr(mat_reg, "MatrixSequence", []):
            raw = mat.FrameOfReferenceTransformationMatrix
            matrix = np.array(raw, dtype=float).reshape(4, 4)

            is_identity = np.allclose(matrix, np.eye(4))
            if is_identity:
                print(f"  Matrix: IDENTITY (this is the FIXED scan — it stays put)")
            else:
                print(f"  Matrix: (this is the MOVING scan — here's how it moved)")
                print(f"  {matrix}")
    print()

--- Registration entry 0 ---
  Frame of Reference UID: ...7361169853.5388955629634219181
  Matrix: (this is the MOVING scan — here's how it moved)
  [[ 9.99987872e-01  3.23536938e-19  4.92495376e-03 -1.29694731e+02]
 [-3.23528805e-19  1.00000000e+00 -2.44817392e-21  7.84420684e+01]
 [-4.92495376e-03  8.54779829e-22  9.99987872e-01  3.11145038e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

--- Registration entry 1 ---
  Frame of Reference UID: ....30000025091716215150700000033
  Matrix: IDENTITY (this is the FIXED scan — it stays put)



## Step 6: Reading the matrix — it's simpler than it looks

The 4x4 matrix looks scary, but it breaks down into two parts:

```
[ rotation  rotation  rotation  | translation_x ]
[ rotation  rotation  rotation  | translation_y ]
[ rotation  rotation  rotation  | translation_z ]
[    0         0         0      |       1        ]
```

- **Translation** (last column, top 3 rows): how far the dog shifted in mm along x, y, z
- **Rotation** (top-left 3x3 block): how the dog rotated

For a rigid body (dog on a table), that's all you need — shift + rotate.
Let's extract those parts.

In [23]:
# Get the non-identity matrix (the actual transform)
for reg in ds.RegistrationSequence:
    for mat_reg in getattr(reg, "MatrixRegistrationSequence", []):
        for mat in getattr(mat_reg, "MatrixSequence", []):
            raw = mat.FrameOfReferenceTransformationMatrix
            m = np.array(raw, dtype=float).reshape(4, 4)
            if not np.allclose(m, np.eye(4)):
                transform_matrix = m

translation = transform_matrix[:3, 3]
rotation = transform_matrix[:3, :3]

print("Translation (how far the dog shifted, in mm):")
print(f"  x: {translation[0]:.1f} mm")
print(f"  y: {translation[1]:.1f} mm")
print(f"  z: {translation[2]:.1f} mm")
print()

# Check if rotation is close to identity (no rotation) or significant
from scipy.spatial.transform import Rotation
angles = Rotation.from_matrix(rotation).as_euler("xyz", degrees=True)
print("Rotation (how much the dog rotated, in degrees):")
print(f"  around x: {angles[0]:.2f}°")
print(f"  around y: {angles[1]:.2f}°")
print(f"  around z: {angles[2]:.2f}°")

Translation (how far the dog shifted, in mm):
  x: -129.7 mm
  y: 78.4 mm
  z: 311.1 mm

Rotation (how much the dog rotated, in degrees):
  around x: 0.00°
  around y: 0.28°
  around z: -0.00°


## Step 7: How does "Frame of Reference" tie things together?

This is the glue. Every CT series lives in a "Frame of Reference" — think of it as
a coordinate system. The planning CT has its own frame. Each CBCT has its own frame.

The RE file says: "here's how to go from frame A to frame B."

Let's see this concretely by matching the Frame of Reference UIDs.

In [24]:
# Every CT series has a Frame of Reference UID. Let's build a lookup.
frame_to_series = {}
for uid, files in series.items():
    ds_sample = pydicom.dcmread(files[0], stop_before_pixels=True)
    frame_uid = str(ds_sample.FrameOfReferenceUID)
    name = getattr(ds_sample, "SeriesDescription", "") or ds_sample.SeriesDate
    frame_to_series[frame_uid] = name

# Now look at our RE file's registration entries
print("The RE file says:\n")
for i, reg in enumerate(ds.RegistrationSequence):
    frame_uid = str(reg.FrameOfReferenceUID)
    series_name = frame_to_series.get(frame_uid, "unknown / not on disk")

    for mat_reg in getattr(reg, "MatrixRegistrationSequence", []):
        for mat in getattr(mat_reg, "MatrixSequence", []):
            raw = mat.FrameOfReferenceTransformationMatrix
            m = np.array(raw, dtype=float).reshape(4, 4)
            role = "FIXED (reference)" if np.allclose(m, np.eye(4)) else "MOVING (gets transformed)"

    print(f"  Entry {i}: {role}")
    print(f"    Frame of Reference -> matches CT series: '{series_name}'")
    print()

The RE file says:

  Entry 0: MOVING (gets transformed)
    Frame of Reference -> matches CT series: '20250925'

  Entry 1: FIXED (reference)
    Frame of Reference -> matches CT series: 'ST POST'



## Step 8: Let's see ALL the registrations for this patient

Now you have the intuition. Let's loop through all 4 RE files and see the full picture.

In [25]:
from scipy.spatial.transform import Rotation

for re_path in sorted(patient_dir.glob("RE.*.dcm")):
    ds = pydicom.dcmread(re_path)
    n_ref = len(getattr(ds, "ReferencedSeriesSequence", []))

    print(f"RE file: ...{re_path.name[3:30]}...")
    print(f"  References {n_ref} CT series")

    # Find the non-identity matrix
    fixed_frame = moving_frame = None
    transform = None
    for reg in ds.RegistrationSequence:
        frame_uid = str(reg.FrameOfReferenceUID)
        for mat_reg in getattr(reg, "MatrixRegistrationSequence", []):
            for mat in getattr(mat_reg, "MatrixSequence", []):
                raw = mat.FrameOfReferenceTransformationMatrix
                m = np.array(raw, dtype=float).reshape(4, 4)
                if np.allclose(m, np.eye(4)):
                    fixed_frame = frame_uid
                else:
                    moving_frame = frame_uid
                    transform = m

    fixed_name = frame_to_series.get(fixed_frame, "unknown")
    moving_name = frame_to_series.get(moving_frame, "not on disk (different timepoint?)")

    print(f"  Fixed (reference):  '{fixed_name}'")
    print(f"  Moving (to align):  '{moving_name}'")

    if transform is not None:
        t = transform[:3, 3]
        angles = Rotation.from_matrix(transform[:3, :3]).as_euler("xyz", degrees=True)
        print(f"  Shift:    x={t[0]:+.1f}mm  y={t[1]:+.1f}mm  z={t[2]:+.1f}mm")
        print(f"  Rotation: x={angles[0]:+.2f}°  y={angles[1]:+.2f}°  z={angles[2]:+.2f}°")
    print()

RE file: ...1.2.246.352.205.52925244484...
  References 1 CT series
  Fixed (reference):  'ST POST'
  Moving (to align):  'not on disk (different timepoint?)'
  Shift:    x=+56.2mm  y=+11.9mm  z=-62.5mm
  Rotation: x=+1.53°  y=-8.50°  z=-0.05°

RE file: ...1.2.246.352.61.10.505415499...
  References 2 CT series
  Fixed (reference):  'ST POST'
  Moving (to align):  '20250925'
  Shift:    x=-129.7mm  y=+78.4mm  z=+311.1mm
  Rotation: x=+0.00°  y=+0.28°  z=-0.00°

RE file: ...1.2.246.352.61.10.506890690...
  References 2 CT series
  Fixed (reference):  'ST POST'
  Moving (to align):  '20250924'
  Shift:    x=-125.6mm  y=+79.9mm  z=+314.6mm
  Rotation: x=+0.00°  y=+1.36°  z=+0.00°

RE file: ...1.2.246.352.61.10.557896917...
  References 2 CT series
  Fixed (reference):  'ST POST'
  Moving (to align):  '20250926'
  Shift:    x=-130.8mm  y=+78.5mm  z=+313.9mm
  Rotation: x=+0.00°  y=+1.34°  z=+0.00°



## What you should see

You should have **4 RE files**:
- **3 of them** connect a CBCT (86 slices) to the planning CT "ST PRE". These are the useful ones — they tell us how to align each treatment-day CBCT to the planning CT.
- **1 of them** references only 1 series and the moving frame doesn't match any CT on disk. That one maps to a different timepoint's planning CT (probably in the `CT_at_diff_timepoint` folder). We'll skip it for now.

Notice the translations are large (~100-300mm) — that's because the CBCT and planning CT use totally different coordinate origins, not because the dog moved that far!

The rotations are small (a few degrees) — that's the actual patient positioning difference.

## Step 9: Now let's actually LOAD a CT volume and look at it

We'll use SimpleITK to read a DICOM series into a 3D array.
SimpleITK handles all the tricky stuff (sorting slices, building the 3D grid) for us.

In [26]:
import SimpleITK as sitk

# SimpleITK can find all DICOM series in a folder automatically
all_series_ids = sitk.ImageSeriesReader.GetGDCMSeriesIDs(str(patient_dir))
print(f"GDCM found {len(all_series_ids)} series in the folder")
print("(this includes non-CT series like RE, RS, etc. — we'll filter those)\n")

# Load the planning CT (ST PRE — the big one)
# Find it by matching our known series UID
planning_uid = None
for uid, files in series.items():
    ds_check = pydicom.dcmread(files[0], stop_before_pixels=True)
    if getattr(ds_check, "SeriesDescription", "") == "ST PRE":
        planning_uid = uid
        break

# Get the file list from SimpleITK (it sorts slices by position automatically)
planning_files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(patient_dir), planning_uid)

reader = sitk.ImageSeriesReader()
reader.SetFileNames(planning_files)
planning_vol = reader.Execute()

print(f"Planning CT loaded!")
print(f"  Size: {planning_vol.GetSize()} (x, y, z voxels)")
print(f"  Spacing: {planning_vol.GetSpacing()} mm (x, y, z)")
print(f"  Origin: {planning_vol.GetOrigin()} mm")

GDCM found 10 series in the folder
(this includes non-CT series like RE, RS, etc. — we'll filter those)

Planning CT loaded!
  Size: (512, 512, 239) (x, y, z voxels)
  Spacing: (0.67578125, 0.67578125, 2.0) mm (x, y, z)
  Origin: (183.662109375, 345.662109375, -44.1) mm


In [27]:
# Now load one CBCT
# Pick the first 86-slice series we find
cbct_uid = None
for uid, files in series.items():
    ds_check = pydicom.dcmread(files[0], stop_before_pixels=True)
    if len(files) == 86:
        cbct_uid = uid
        cbct_date = ds_check.SeriesDate
        break

cbct_files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(patient_dir), cbct_uid)
reader = sitk.ImageSeriesReader()
reader.SetFileNames(cbct_files)
cbct_vol = reader.Execute()

print(f"CBCT loaded! (from {cbct_date})")
print(f"  Size: {cbct_vol.GetSize()} (x, y, z voxels)")
print(f"  Spacing: {cbct_vol.GetSpacing()} mm")
print(f"  Origin: {cbct_vol.GetOrigin()} mm")
print()
print("Notice: the origins are very different! That's why the translations in the RE file are so large.")

CBCT loaded! (from 20250926)
  Size: (512, 512, 86) (x, y, z voxels)
  Spacing: (0.48828125, 0.48828125, 2.0) mm
  Origin: (249.755859375, 249.755859375, -86.0) mm

Notice: the origins are very different! That's why the translations in the RE file are so large.


## Step 10: Apply the transform — the actual alignment

Here's where it all comes together. The RE file gave us a matrix that maps
**moving (CBCT) coordinates -> fixed (planning CT) coordinates.**

To "resample" the CBCT into the planning CT's space, SimpleITK needs the
**reverse direction**: for each point in the output (planning CT grid),
where does it come from in the input (CBCT)?

So we just invert the matrix. That's it.

In [28]:
# First, we need to find the right transform for this specific CBCT.
# We need the RE file whose moving frame matches this CBCT's frame of reference.
cbct_sample = pydicom.dcmread(cbct_files[0], stop_before_pixels=True)
cbct_frame = str(cbct_sample.FrameOfReferenceUID)

# Search RE files for the one that has this frame
the_matrix = None
for re_path in sorted(patient_dir.glob("RE.*.dcm")):
    ds = pydicom.dcmread(re_path)
    for reg in ds.RegistrationSequence:
        if str(reg.FrameOfReferenceUID) == cbct_frame:
            for mat_reg in getattr(reg, "MatrixRegistrationSequence", []):
                for mat in getattr(mat_reg, "MatrixSequence", []):
                    raw = mat.FrameOfReferenceTransformationMatrix
                    m = np.array(raw, dtype=float).reshape(4, 4)
                    if not np.allclose(m, np.eye(4)):
                        the_matrix = m
                        print(f"Found matching RE file: {re_path.name}")

print(f"\nTransform matrix (CBCT -> planning CT):")
print(the_matrix)

Found matching RE file: RE.1.2.246.352.61.10.5578969175256111695.12197233281027642784.dcm

Transform matrix (CBCT -> planning CT):
[[ 9.99726893e-01  0.00000000e+00  2.33696386e-02 -1.30842026e+02]
 [ 0.00000000e+00  1.00000000e+00  0.00000000e+00  7.84569629e+01]
 [-2.33696386e-02  0.00000000e+00  9.99726893e-01  3.13907507e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [29]:
# Invert it — SimpleITK needs fixed->moving, the RE file gives us moving->fixed
inv_matrix = np.linalg.inv(the_matrix)

# Build a SimpleITK transform from the inverted matrix
transform = sitk.AffineTransform(3)
transform.SetMatrix(inv_matrix[:3, :3].flatten().tolist())
transform.SetTranslation(inv_matrix[:3, 3].tolist())

# Resample! This puts the CBCT into the planning CT's coordinate grid.
# - planning_vol defines the output grid (size, spacing, origin)
# - cbct_vol is the input being resampled
# - transform tells SimpleITK where to sample from for each output voxel
# - sitkLinear = smooth interpolation
# - -1000.0 = fill value for air (in Hounsfield Units) where CBCT has no data
aligned_cbct = sitk.Resample(cbct_vol, planning_vol, transform, sitk.sitkLinear, -1000.0)

print("Alignment done!")
print(f"  Output size: {aligned_cbct.GetSize()} (same as planning CT)")
print(f"  Output origin: {aligned_cbct.GetOrigin()} (same as planning CT)")

Alignment done!
  Output size: (512, 512, 239) (same as planning CT)
  Output origin: (183.662109375, 345.662109375, -44.1) (same as planning CT)


In [30]:
# Save all three so you can compare in a viewer
output_dir = Path("../output")
output_dir.mkdir(exist_ok=True)

sitk.WriteImage(planning_vol, str(output_dir / "planning_ct.nii.gz"))
sitk.WriteImage(cbct_vol, str(output_dir / "original_cbct.nii.gz"))
sitk.WriteImage(aligned_cbct, str(output_dir / "aligned_cbct.nii.gz"))

print("Saved 3 files to output/:")
print("  planning_ct.nii.gz   — the reference scan")
print("  original_cbct.nii.gz — CBCT before alignment (different coordinate space)")
print("  aligned_cbct.nii.gz  — CBCT after alignment (same coordinate space as planning CT)")

Saved 3 files to output/:
  planning_ct.nii.gz   — the reference scan
  original_cbct.nii.gz — CBCT before alignment (different coordinate space)
  aligned_cbct.nii.gz  — CBCT after alignment (same coordinate space as planning CT)


## Recap — what just happened

1. **Discovered CT series** — found 5 series in the folder (2 planning, 3 CBCT)
2. **Read an RE file** — found it references 2 series and stores a 4x4 rigid matrix
3. **Understood the matrix** — it's a rotation + translation that maps CBCT coords to planning CT coords
4. **Matched via Frame of Reference** — each scan has a coordinate frame ID, the RE file links them
5. **Loaded volumes with SimpleITK** — DICOM slices become a 3D image with proper spacing/origin
6. **Applied the transform** — inverted the matrix (SimpleITK needs fixed->moving), resampled the CBCT onto the planning CT grid